# Task 4: Graph Topology Ingestion into Neo4j

### Mục tiêu
- Kết nối **Neo4j Kafka Connector Sink** với hai topic `cpg-nodes` và `cpg-edges`.
- Ghi trực tiếp cấu trúc đồ thị từ **Kafka** vào **Neo4j**, không thông qua Spark.
- Sử dụng `MERGE` thay cho `CREATE` để đảm bảo **idempotent**, tức là khi xử lý lại dữ liệu sẽ không tạo bản ghi trùng.

### Kiến trúc

```text
cpg_parser.py  -->  Kafka (cpg-nodes, cpg-edges)  -->  Kafka Connect  -->  Neo4j
                    localhost:9092                      :8083               :7687
```

### Docker Compose

| Dịch vụ | Image | Cổng |
|:---|:---|:---:|
| ZooKeeper | confluentinc/cp-zookeeper:7.6.1 | 2181 |
| Kafka | confluentinc/cp-kafka:7.6.1 | 9092 |
| Kafka Connect | confluentinc/cp-kafka-connect:7.6.1 | 8083 |
| Neo4j | neo4j:5.20.0-community | 7474, 7687 |


### 0. Cấu hình đường dẫn

In [16]:

import subprocess as _sp
from pathlib import Path

# Chuyển đường dẫn Windows sang định dạng WSL
def to_wsl_path(p):
    p = str(p).replace("\\", "/")
    if len(p) >= 2 and p[1] == ":":
        return f"/mnt/{p[0].lower()}{p[2:]}"
    return p

# Tìm Python của Conda trong WSL
def find_conda_wsl():
    for name in ["miniconda3", "anaconda3", "miniforge3"]:
        path = f"~/{name}/bin/python3"
        r = _sp.run(
            ["wsl", "-e", "bash", "-c", f"ls {path}"],
            capture_output=True,
            text=True,
        )
        if r.returncode == 0:
            return r.stdout.strip()

    # Nếu không tìm thấy thì dùng Python mặc định
    return "python3"


WSL_ROOT = to_wsl_path(PROJECT_ROOT)
TASK4_WSL = f"{WSL_ROOT}/src/task4"
CONDA_WSL = find_conda_wsl()

print(f"WSL_ROOT  = {WSL_ROOT}")
print(f"TASK4_WSL = {TASK4_WSL}")
print(f"CONDA_WSL = {CONDA_WSL}")

def run_wsl(bash_cmd, timeout=180):
    """Chạy lệnh bash trong WSL, trả về CompletedProcess."""
    return subprocess.run(
        ["wsl", "-e", "bash", "-c", bash_cmd],
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        timeout=timeout,
    )

WSL_ROOT  = /mnt/e/BD
TASK4_WSL = /mnt/e/BD/src/task4
CONDA_WSL = /home/myha/miniconda3/bin/python3


### 1. Khởi động các dịch vụ của Task 4

In [17]:
def start_task4_services():
    r = run_wsl(f'{CONDA_WSL} {TASK4_WSL}/start.py', timeout=400)
    if r.stdout: print(r.stdout)
    if r.returncode != 0: print('STDERR:', r.stderr[:500])

start_task4_services()

Kiểm tra và xóa container cũ (nếu có)...
  Đã xóa container: cpg-zookeeper (trạng thái trước đó: running)
  Đã xóa container: cpg-kafka (trạng thái trước đó: running)
  Đã xóa container: cpg-kafka-connect (trạng thái trước đó: running)
  Đã xóa container: cpg-neo4j (trạng thái trước đó: running)
Khởi động Docker Stack...
  Container cpg-zookeeper Started
  Container cpg-neo4j Started
  Container cpg-kafka Started
  Container cpg-kafka-connect Started
Chờ Kafka Broker sẵn sàng (15 giây)...
Tạo các Kafka topic...
  cpg-nodes: Created topic cpg-nodes.
  cpg-edges: Created topic cpg-edges.
  cpg-metadata: Created topic cpg-metadata.
  cpg-errors: Created topic cpg-errors.
Chờ Kafka Connect và plugin Neo4j khởi động (có thể mất 3–5 phút)...
  [1/36] Chờ 5 giây...
  [2/36] Chờ 5 giây...
  [3/36] Chờ 5 giây...
  [4/36] Chờ 5 giây...
  Đã tìm thấy plugin Neo4j: org.neo4j.connectors.kafka.sink.Neo4jConnector
Docker Stack và Kafka Connect đã sẵn sàng.



### 2. Đăng ký Neo4j Connector Sink

Sử dụng Kafka Connect REST API để đăng ký hai connector:

- `neo4j-cpg-nodes-sink`: đọc dữ liệu từ topic `cpg-nodes` và ghi các node vào Neo4j bằng câu lệnh `MERGE`.
- `neo4j-cpg-edges-sink`: đọc dữ liệu từ topic `cpg-edges` và ghi các relationship vào Neo4j bằng câu lệnh `MERGE`.

In [18]:
def register_neo4j_connectors():
    r = run_wsl(f'{CONDA_WSL} {TASK4_WSL}/setup.py')
    if r.stdout: print(r.stdout)
    if r.returncode != 0: print('STDERR:', r.stderr[:500])

register_neo4j_connectors()

Đang chờ Kafka Connect sẵn sàng... OK

Kiểm tra plugin Neo4j:
  Tìm thấy: org.neo4j.connectors.kafka.sink.Neo4jConnector (phiên bản 5.1.10)
  Tìm thấy: org.neo4j.connectors.kafka.source.Neo4jConnector (phiên bản 5.1.10)

Đăng ký các connector...
  [neo4j-cpg-nodes-sink] Đăng ký thành công
  [neo4j-cpg-edges-sink] Đăng ký thành công

Chờ connector khởi động (30 giây)...

Trạng thái connector:
  neo4j-cpg-nodes-sink: RUNNING, tasks=['RUNNING']
  neo4j-cpg-edges-sink: RUNNING, tasks=['RUNNING']



### 3. Gửi CPG Events lên Kafka

Chạy `cpg_parser.py` với 5 tệp Python đầu tiên để phát (emit) các sự kiện lên các Kafka topic sau:

- `cpg-nodes`: chứa các node của Code Property Graph (AST, CFG, DFG).
- `cpg-edges`: chứa các cạnh giữa các node, bao gồm quan hệ AST (cha–con), CFG (luồng điều khiển), DFG (luồng dữ liệu) và `CALL`.
- `cpg-metadata`: chứa thông tin metadata của từng tệp Python được xử lý.

In [19]:
def emit_cpg_events():
    r = run_wsl(f'{CONDA_WSL} {TASK4_WSL}/emit.py', timeout=300)
    if r.stdout: print(r.stdout)
    if r.returncode != 0: print('STDERR:', r.stderr[:500])

emit_cpg_events()

=== Starting CPG Parser Service ===
Discovered CSV : /mnt/e/BD/output/discovered_files.csv
Repo Root      : /mnt/e/BD/peft
Kafka Brokers  : localhost:9092
Schema Version : 1.0.0
Dry Run Mode   : False

Found 5 source files to parse.
[OK] Connected to Kafka Producer successfully.

[1/5] Processing: docs/source/_config.py ... Parsed and Sent (Nodes: 5, Edges: 6)
[2/5] Processing: examples/adamss_finetuning/glue_adamss_asa_example.py ... Parsed and Sent (Nodes: 2222, Edges: 3935)
[3/5] Processing: examples/adamss_finetuning/glue_adamss_asa_manual_example.py ... Parsed and Sent (Nodes: 2175, Edges: 3841)
[4/5] Processing: examples/adamss_finetuning/image_classification_adamss_asa.py ... Parsed and Sent (Nodes: 2319, Edges: 4123)
[5/5] Processing: examples/alora_finetuning/alora_finetuning.py ... Parsed and Sent (Nodes: 1227, Edges: 2045)

=== Parser Run Summary ===
Total Source Files processed : 5
Successfully Parsed          : 5
Failed                       : 0
Total Nodes emitted        

### 4. Kiểm chứng dữ liệu đã được đưa vào Neo4j

Sau khi các sự kiện được gửi lên Kafka, Kafka Connect sẽ tự động đọc dữ liệu từ các topic và ghi vào Neo4j.

Cell dưới đây sẽ chờ khoảng **90 giây** để Kafka Connect xử lý dữ liệu theo từng lô (batch), sau đó truy vấn Neo4j và hiển thị kết quả.

**Các nội dung cần kiểm tra:**

1. Số lượng `CpgNode` lớn hơn 0.
2. Số lượng `CPG_EDGE` lớn hơn 0, bao gồm các loại quan hệ: `AST`, `CFG`, `DFG` và `CALL`.
3. `duplicates = 0`, xác nhận câu lệnh `MERGE` hoạt động đúng và không tạo dữ liệu trùng lặp khi xử lý lại.

In [20]:
def verify_neo4j_ingestion():
    r = run_wsl(f'{CONDA_WSL} {TASK4_WSL}/verify.py')
    if r.stdout: print(r.stdout)
    if r.returncode != 0: print('STDERR:', r.stderr[:500])

verify_neo4j_ingestion()

Tổng số CpgNode  : 15,148
Tổng số CPG_EDGE : 26,780

Thống kê loại node (Top 10):
  Load                         3,631
  Name                         3,019
  Constant                     2,002
  Attribute                    949
  Call                         915
  keyword                      719
  Store                        646
  Expr                         424
  Assign                       396
  Dict                         256

Thống kê loại relationship:
  AST                          15,138
  CFG                          9,791
  DFG                          1,106
  CALL                         745

Các tệp đã xử lý:
  examples/adamss_finetuning/image_classification_adamss_asa.py  ->  4,422 node
  examples/adamss_finetuning/glue_adamss_asa_example.py  ->  4,212 node
  examples/adamss_finetuning/glue_adamss_asa_manual_example.py  ->  4,129 node
  examples/alora_finetuning/alora_finetuning.py  ->  2,375 node
  docs/source/_config.py  ->  10 node

Node mẫu:
{
  "schema_version": "

### 5. Kết quả

**1. Triển khai hạ tầng**  
Kiểm tra trạng thái các container trong stack:

<div align="center">
  <img src="../task4/1_v2.png" alt="Docker status" width="800"/>
  <p><i>Hình 4.5.1: Các dịch vụ Docker (Zookeeper, Kafka, Connect, Neo4j) đang hoạt động (Up)</i></p>
</div>  

**2. Trạng thái Connectors**  
Xác nhận các sink connector đã được đăng ký và đang chạy:

<div align="center">
  <img src="../task4/2_v2.png" alt="Connector status" width="800"/>
  <p><i>Hình 4.5.2: Trạng thái RUNNING của neo4j-cpg-nodes-sink và neo4j-cpg-edges-sink</i></p>
</div>  

**3. Kiểm chứng dữ liệu trong Neo4j**  
Dưới đây là kết quả thống kê số lượng Nodes và Relationships đã được ingestion thành công:

<div align="center">
  <img src="../task4/3.2_v2.png" alt="Neo4j Nodes" width="800"/>
  <p><i>Hình 4.5.3: Kết quả truy vấn đếm CpgNode</i></p>
</div>

<div align="center">
  <img src="../task4/3.1_v2.png" alt="Neo4j Edges" width="800"/>
  <p><i>Hình 4.5.4: Kết quả truy vấn đếm CPG_EDGE</i></p>
</div>

### 6. Reflection

**What worked**
- Docker Compose giúp khởi động toàn bộ stack (ZooKeeper, Kafka, Kafka Connect và Neo4j) chỉ với một lệnh, giúp việc thiết lập môi trường nhanh và thuận tiện.
- Neo4j Kafka Connector Sink tự động đọc dữ liệu từ các topic `cpg-nodes` và `cpg-edges`, sau đó thực thi câu lệnh Cypher để ghi trực tiếp vào Neo4j mà không cần thông qua Spark.
- Sử dụng `MERGE` thay cho `CREATE` đảm bảo tính idempotent. Sau khi emit dữ liệu nhiều lần, số lượng node và relationship không thay đổi, đồng thời `duplicates = 0`.

**What failed**
- Neo4j Connector phiên bản 5.x thay đổi khóa cấu hình từ `neo4j.topic.cypher.<name>` sang `neo4j.cypher.topic.<name>`, khiến connector không hoạt động khi sử dụng cấu hình của phiên bản cũ.
- Connector ghi `cpg-edges` chỉ hoạt động khi các node tương ứng đã tồn tại trong Neo4j. Nếu node connector chưa xử lý xong, các relationship sẽ không được tạo.
- Đường dẫn tệp sinh trên Windows sử dụng dấu `\`, trong khi các script chạy trên WSL yêu cầu định dạng `/`, dẫn đến lỗi không tìm thấy tệp.

**How resolved**
- Cập nhật cấu hình connector theo đúng cú pháp của Neo4j Connector 5.x và kiểm tra plugin thông qua Kafka Connect REST API trước khi đăng ký connector.
- Khởi động Kafka Connect trước, chờ connector sẵn sàng rồi mới emit dữ liệu để đảm bảo các node được ghi vào Neo4j trước khi xử lý relationship.
- Chuyển đổi đường dẫn từ định dạng Windows sang định dạng WSL trước khi truyền vào các script, giúp các bước xử lý hoạt động ổn định trên cả hai môi trường.